# 03 — Sequential vs Parallel Scan

The selective scan recurrence $h_t = a_t \cdot h_{t-1} + b_t$ is inherently sequential — each step depends on the previous one. This is a problem on GPUs, which are designed for massive parallelism.

**What you will learn:**
1. Why the affine recurrence is associative
2. The Hillis-Steele parallel scan algorithm
3. The chunked scan as a practical middle ground
4. When parallelism actually helps (and when it doesn't)


## The Associativity Trick

The recurrence $h_t = a_t \cdot h_{t-1} + b_t$ is an *affine transform*. The composition of two affine transforms is also affine:

If transform 1 is $x \mapsto a_1 x + b_1$ and transform 2 is $x \mapsto a_2 x + b_2$, then applying 1 then 2 gives:

$$x \mapsto a_2(a_1 x + b_1) + b_2 = (a_2 a_1) x + (a_2 b_1 + b_2)$$

This composition is **associative** (but not commutative), which means we can use parallel prefix-sum algorithms to compute all $h_t$ values simultaneously.

The `_combine` function in `parallel_scan.py` implements this:
```python
def _combine(left_a, left_b, right_a, right_b):
    return right_a * left_a, right_a * left_b + right_b
```


## Three Scan Implementations

| Algorithm | Work | Span | Memory | Best for |
|-----------|------|------|--------|----------|
| Sequential | $O(L)$ | $O(L)$ | $O(1)$ state | CPU, short sequences |
| Hillis-Steele | $O(L \log L)$ | $O(\log L)$ | $O(L)$ | Maximum parallelism |
| Chunked | $O(L)$ | $O(C + L/C)$ | $O(C)$ per chunk | Practical GPU balance |

The chunked scan is what matters in practice: it processes chunks of size $C$ sequentially (small enough to be fast), then uses the associative property to propagate carries between chunks.


In [ ]:
import torch
import time
from mamba_minimal.parallel_scan import (
    sequential_affine_scan,
    hillis_steele_affine_scan,
    chunked_affine_scan,
)

torch.manual_seed(42)

# Correctness check on a small example
B, L, F = 4, 1024, 16
a = torch.rand(B, L, F)
b = torch.randn(B, L, F)

ref = sequential_affine_scan(a, b)
par = hillis_steele_affine_scan(a, b)
chk = chunked_affine_scan(a, b, chunk_size=128)

print("=== Correctness (vs sequential reference) ===")
print(f"Hillis-Steele max error: {(ref - par).abs().max().item():.2e}")
print(f"Chunked (C=128) max error: {(ref - chk).abs().max().item():.2e}")

## Timing Comparison

Let us measure wall-clock time for each algorithm across different sequence lengths:


In [ ]:
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
lengths = [64, 128, 256, 512, 1024, 2048, 4096]
B_bench, F_bench = 4, 32

results = {name: [] for name in ["sequential", "hillis-steele", "chunked-64", "chunked-256"]}

for L in lengths:
    a = torch.rand(B_bench, L, F_bench, device=device)
    b = torch.randn(B_bench, L, F_bench, device=device)
    
    fns = {
        "sequential": lambda: sequential_affine_scan(a, b),
        "hillis-steele": lambda: hillis_steele_affine_scan(a, b),
        "chunked-64": lambda: chunked_affine_scan(a, b, chunk_size=64),
        "chunked-256": lambda: chunked_affine_scan(a, b, chunk_size=256),
    }
    
    for name, fn in fns.items():
        # warmup
        for _ in range(3):
            _ = fn()
        if device == "cuda":
            torch.cuda.synchronize()
        
        timings = []
        for _ in range(10):
            start = time.perf_counter()
            _ = fn()
            if device == "cuda":
                torch.cuda.synchronize()
            timings.append((time.perf_counter() - start) * 1000)
        
        results[name].append(sorted(timings)[len(timings)//2])  # median

fig, ax = plt.subplots(figsize=(10, 5))
for name, times in results.items():
    ax.plot(lengths, times, marker="o", label=name, linewidth=2)

ax.set_xlabel("Sequence length")
ax.set_ylabel("Latency (ms)")
ax.set_title(f"Scan Algorithm Comparison (B={B_bench}, F={F_bench}, device={device})")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
fig.tight_layout()
plt.show()

## Numerical Precision

The parallel scan does more floating-point operations (each combine step involves multiplies and adds). Let us check how error accumulates with sequence length:


In [ ]:
lengths_prec = [32, 64, 128, 256, 512, 1024, 2048, 4096]
errors_hs = []
errors_chk = []

for L in lengths_prec:
    a = torch.rand(2, L, 8, dtype=torch.float32)
    b = torch.randn(2, L, 8, dtype=torch.float32)
    
    ref = sequential_affine_scan(a, b)
    par = hillis_steele_affine_scan(a, b)
    chk = chunked_affine_scan(a, b, chunk_size=64)
    
    errors_hs.append((ref - par).abs().max().item())
    errors_chk.append((ref - chk).abs().max().item())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lengths_prec, errors_hs, "o-", label="Hillis-Steele", linewidth=2)
ax.plot(lengths_prec, errors_chk, "s-", label="Chunked (C=64)", linewidth=2)
ax.set_xlabel("Sequence length")
ax.set_ylabel("Max absolute error vs sequential")
ax.set_title("Numerical Error Accumulation (FP32)")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f"At L=4096: Hillis-Steele error = {errors_hs[-1]:.2e}, Chunked error = {errors_chk[-1]:.2e}")

## Chunked Scan: How It Works

The chunked scan is the most practical algorithm for GPUs:

```
Sequence:  [chunk 0] [chunk 1] [chunk 2] [chunk 3] ...

Step 1: Process each chunk independently with sequential scan
         -> get local outputs and per-chunk carries (a_carry, b_carry)

Step 2: Scan the carries across chunks (sequential over ~L/C elements)
         -> get the correct starting state for each chunk

Step 3: Adjust each chunk's outputs using the corrected prefix
```

This gives us $O(L)$ work with parallelism within each chunk, and the cross-chunk scan is only over $L/C$ elements.


## SSD Connection

The Mamba-2 paper introduces the **State Space Duality (SSD)** view, which shows that the within-chunk computation can be expressed as a masked matrix multiply:

$$y_{\text{chunk}} = M \cdot b_{\text{chunk}}$$

where $M$ is a lower-triangular decay matrix. This connects the recurrent and attention views of state space models.

See `src/mamba_minimal/ssd.py` for a minimal implementation of this idea.


In [ ]:
from mamba_minimal.ssd import check_ssd_matches_sequential

torch.manual_seed(0)
a = torch.rand(2, 256, 8).clamp(0.01, 0.99)
b = torch.randn(2, 256, 8)

max_err = check_ssd_matches_sequential(a, b, chunk_size=64)
print(f"SSD vs sequential max error: {max_err:.2e}")

## Key Takeaways

1. The affine recurrence is **associative**, enabling parallel prefix scans
2. **Hillis-Steele** maximizes parallelism but does $O(L \log L)$ work
3. **Chunked scan** is the practical choice: $O(L)$ work with tunable chunk size
4. Numerical error grows with sequence length but stays manageable in FP32
5. The SSD view (Mamba-2) reframes within-chunk computation as masked matmul

**Next notebook:** We look at the Triton fused kernel that puts the scan on GPU with minimal memory traffic.
